**Chargement du CSV :**,

In [ ]:
df = spark.read.option("header", "true") \
               .option("inferSchema", "true") \
               .csv("Files/bronze/supply_chain_data.csv")

df.printSchema()
print(f"Lignes : {df.count()} | Colonnes : {len(df.columns)}")

**Aperçu des données :**

In [ ]:
display(df.limit(10))

**Analyse qualité (valeurs nulles) :**

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, isnan, when, count

null_counts = df.select([
    spark_sum(when(col(c).isNull() | isnan(c), 1).otherwise(0)).alias(c)
    for c in df.columns
])
display(null_counts)

**Distribution des types de produits :**

In [ ]:
display(
    df.groupBy("Product type")
      .count()
      .orderBy("count", ascending=False)
)

**Enlever les espaces dans les noms des colonnes:**
Car des colonnes mal nommées peuvent gêner lors des transformations futures

In [ ]:
df = df.toDF(*[c.replace(" ", "_") for c in df.columns])
display(df.limit(3))

**Sauvegarder en Delta Table (Bronze officielle)**
Transformation du fichier CSV brut en Delta Table requêtable

In [ ]:
df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("bronze_supply_chain_raw")